In [1]:
import pandas as pd
import numpy as np
import pickle

import warnings

from statsmodels.tsa.api import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [2]:
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("../../data/pre_data.csv")

In [4]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,8460.0
1,Gạo XK 5% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,10070.0
2,Gạo XK 5% tấm,Tiền Giang,Thương lái thu mua,CTV địa phương,2025-05-14,15300.0
3,ST24,Hậu Giang,Thương lái thu mua,CTV địa phương,2025-05-13,10650.0
4,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-09,8520.0


In [5]:
df[df["Tên_mặt_hàng"] == "Gạo NL 25% tấm"]

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,8460.0
4,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-09,8520.0
9,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-04-25,8510.0
24,Gạo NL 25% tấm,Kiên Giang,Thu mua,Giồng Riềng,2024-12-20,9930.0
26,Gạo NL 25% tấm,Kiên Giang,Thu mua,Giồng Riềng,2024-12-18,9980.0
...,...,...,...,...,...,...
891,Gạo NL 25% tấm,Kiên Giang,Khác,CTV địa phương,2023-01-12,9600.0
896,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2023-01-11,9800.0
902,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2023-01-06,9600.0
912,Gạo NL 25% tấm,Kiên Giang,Khác,CTV địa phương,2023-01-05,9600.0


In [6]:
for col in ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]:
    lbl_encoder = LabelEncoder()
    df[col] = lbl_encoder.fit_transform(df[col])
    print(lbl_encoder.classes_)

    with open(f"../train/lbl_scaler/{col}.pkl", "wb") as file:
        pickle.dump(lbl_encoder, file)

['Bưởi da xanh' 'Bưởi năm roi' 'Bắp cải' 'Bồn bồn' 'Cam sành (loại 1)'
 'Cam sành (loại 2)' 'Chanh' 'Chuối sim' 'Chôm chôm Java'
 'Chôm chôm đường' 'Cà rốt' 'Dưa chuột/dưa chuội' 'Dưa hấu (loại 2)'
 'Dưa vàng' 'Dừa' 'Gạo Bắc thơm' 'Gạo IR50404' 'Gạo Jasmine' 'Gạo NL 15%'
 'Gạo NL 25% tấm' 'Gạo XK 5% tấm' 'Gạo thơm Đài Loan (trong)' 'Khoai tây'
 'Lúa Bắc thơm' 'Lúa Khang dân' 'Lúa OM 5451' 'Lúa Q5 (lúa tươi)'
 'Lúa ST 24 (lúa tươi)' 'Lúa ST20 (lúa tươi)' 'Lúa ST24'
 'Lúa T10 (lúa tươi)' 'Lúa thường IR 50404 (khô)'
 'Lúa thường IR 50404 (tươi)' 'Mít Thái' 'Măng cụt' 'Mướp' 'Mướp hương'
 'Mướp đắng' 'Mồng tơi' 'Na' 'Na thái' 'Quýt đường' 'Quả bí xanh'
 'Quả bí đỏ' 'Rau cải' 'Rau cải mơ' 'Rau cải ngọt' 'ST24' 'Su su quả'
 'Sầu Riêng Ri 6 (loại xô)' 'Trái dứa/thơm (to)' 'Xoài cát Hòa Lộc'
 'Đu đủ']
['An Giang' 'Bạc Liêu' 'Bến Tre' 'Cà Mau' 'Cần Thơ' 'Hà Nội' 'Hậu Giang'
 'Kiên Giang' 'Sóc Trăng' 'Sơn La' 'Thái Bình' 'Tiền Giang' 'Trà Vinh'
 'Vĩnh Long' 'Đồng Tháp']
['Bán buôn' 'Bán lẻ' 'Khá

In [7]:
for col in ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]:
    scaler = MinMaxScaler()  
    df[col] = scaler.fit_transform(df[[col]])

    with open(f"../train/mm_scaler/{col}.pkl", "wb") as file:
        pickle.dump(scaler, file)

In [8]:
def get_mm_dict():
    cols = ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../mm_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

def get_lbl_dict():
    cols = ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../lbl_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

In [9]:
mm_dict = get_mm_dict()
mm_dict

{'Tên_mặt_hàng': MinMaxScaler(),
 'Thị_trường': MinMaxScaler(),
 'Loại_giá': MinMaxScaler(),
 'Nguồn': MinMaxScaler()}

In [10]:
lbl_dict = get_lbl_dict()
lbl_dict

{'Tên_mặt_hàng': LabelEncoder(),
 'Thị_trường': LabelEncoder(),
 'Loại_giá': LabelEncoder(),
 'Nguồn': LabelEncoder()}

In [ ]:
groups = df.groupby(["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"])
results = []

for keys, group_df in groups:
    group_df = group_df.sort_values("Ngày")
    
    if len(group_df) < 10:
        continue  # Skip if not enough data

    y = group_df["Giá"]
    exog = group_df[["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]]

    try:
        model = SARIMAX(y, exog=exog, order=(1,1,1), seasonal_order=(0,0,0,0))
        model_fit = model.fit(disp=False)
        with open("./sarimax.pkl", "wb") as file:
            pickle.dump(model_fit, file)
        
        # Forecast using the last row of exog
        forecast = model_fit.forecast(steps=1, exog=exog.tail(1))
        results.append((keys, forecast.iloc[0]))

    except Exception as e:
        print(f"Skip group {keys} due to error: {e}")


In [ ]:
groups = df.groupby(["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"])
results = []

for keys, group_df in groups:
    group_df = group_df.sort_values("Ngày")
    
    if len(group_df) < 10:
        continue  # Skip if not enough data

    y = group_df["Giá"]
    exog = group_df[["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]]

    try:
        model = ARIMA(y, exog=exog, order=(1,1,1))
        model_fit = model.fit()
        with open("./arima.pkl", "wb") as file:
            pickle.dump(model_fit, file)
        
        # Forecast using the last row of exog
        forecast = model_fit.forecast(steps=10, exog=exog.tail(1))
        results.append((keys, forecast.iloc[0]))

    except Exception as e:
        print(f"Skip group {keys} due to error: {e}")


Skip group (np.float64(0.0), np.float64(0.3571428571428571), np.float64(0.14285714285714285), np.float64(0.0)) due to error: Provided exogenous values are not of the appropriate shape. Required (10, 4), got (1, 4).
Skip group (np.float64(0.0), np.float64(0.6428571428571428), np.float64(0.14285714285714285), np.float64(0.0)) due to error: Provided exogenous values are not of the appropriate shape. Required (10, 4), got (1, 4).
Skip group (np.float64(0.0), np.float64(0.7857142857142857), np.float64(0.5714285714285714), np.float64(0.0)) due to error: Provided exogenous values are not of the appropriate shape. Required (10, 4), got (1, 4).
Skip group (np.float64(0.0), np.float64(0.7857142857142857), np.float64(0.7142857142857142), np.float64(0.0)) due to error: Provided exogenous values are not of the appropriate shape. Required (10, 4), got (1, 4).
Skip group (np.float64(0.019230769230769232), np.float64(0.7857142857142857), np.float64(0.5714285714285714), np.float64(0.0)) due to error: P